# Домашнее задание HW14: Эмбеддинги, FAISS, оценка retrieval и mini-RAG

В этом ноутбуке реализуем pipeline для работы с базой знаний: загрузка документов, чанкинг, построение эмбеддингов, индекс FAISS, оценка retrieval, обновление базы знаний и простой mini-RAG.

In [13]:
# Установка пакетов если нужно
%pip install faiss-cpu sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# Импорты, seed и среда
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from sentence_transformers import SentenceTransformer
import torch

# Фиксируем seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Устройство
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: cpu


In [15]:
# База знаний и первичный анализ
documents = [
    "Python - это высокоуровневый язык программирования общего назначения. Он поддерживает несколько парадигм программирования, включая объектно-ориентированное, императивное и функциональное программирование.",
    "Переменные в Python используются для хранения данных. Они создаются путем присваивания значения. Python имеет динамическую типизацию, поэтому тип переменной определяется автоматически.",
    "Циклы в Python позволяют выполнять код многократно. Основные типы циклов: for и while. Цикл for используется для итерации по последовательностям.",
    "Функции в Python определяются с помощью ключевого слова def. Они позволяют группировать код для повторного использования. Функции могут принимать параметры и возвращать значения.",
    "Списки в Python - это изменяемые последовательности. Они могут содержать элементы разных типов. Списки создаются с помощью квадратных скобок [].",
    "Словари в Python - это структуры данных, которые хранят пары ключ-значение. Они создаются с помощью фигурных скобок {}. Ключи должны быть неизменяемыми.",
    "Классы в Python используются для создания объектов. Они определяют свойства и методы объектов. Классы создаются с помощью ключевого слова class.",
    "Исключения в Python обрабатываются с помощью блоков try-except. Это позволяет gracefully обрабатывать ошибки во время выполнения программы.",
    "Модули в Python - это файлы, содержащие определения и операторы. Они позволяют организовывать код. Модули импортируются с помощью ключевого слова import.",
    "Ввод и вывод в Python осуществляется с помощью функций input() и print(). Функция input() считывает данные от пользователя, print() выводит данные на экран."
]

print(f"Число документов: {len(documents)}")
print("\nПримеры документов:")
for i in range(3):
    print(f"{i+1}. {documents[i][:100]}...")

Число документов: 10

Примеры документов:
1. Python - это высокоуровневый язык программирования общего назначения. Он поддерживает несколько пара...
2. Переменные в Python используются для хранения данных. Они создаются путем присваивания значения. Pyt...
3. Циклы в Python позволяют выполнять код многократно. Основные типы циклов: for и while. Цикл for испо...


In [16]:
# Чанкинг документов
def chunk_text(text, chunk_size=100, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
        if start >= len(text):
            break
    return chunks

chunks = []
chunk_sources = []
for i, doc in enumerate(documents):
    doc_chunks = chunk_text(doc, chunk_size=50, overlap=20)
    chunks.extend(doc_chunks)
    chunk_sources.extend([i] * len(doc_chunks))

print(f"Число чанков: {len(chunks)}")
print("\nПример чанкинга для первого документа:")
for j, chunk in enumerate(chunk_text(documents[0], 100, 20)):
    print(f"Chunk {j+1}: {chunk}")

Число чанков: 58

Пример чанкинга для первого документа:
Chunk 1: Python - это высокоуровневый язык программирования общего назначения. Он поддерживает несколько пара
Chunk 2: ивает несколько парадигм программирования, включая объектно-ориентированное, императивное и функцион
Chunk 3: еративное и функциональное программирование.


In [17]:
from pathlib import Path

# Создаём папку артефактов
Path("artifacts").mkdir(parents=True, exist_ok=True)

# Загружаем модель эмбеддингов
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# Строим эмбеддинги для чанков
chunk_embeddings = model.encode(chunks, convert_to_tensor=False)
chunk_embeddings = np.array(chunk_embeddings)
faiss.normalize_L2(chunk_embeddings)

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(chunk_embeddings)

print(f"Размерность эмбеддингов: {dimension}")
print(f"Число векторов в индексе: {index.ntotal}")

# Пример поиска
query = "Что такое переменные в Python?"
query_emb = model.encode([query], convert_to_tensor=False)
query_emb = np.array(query_emb)
faiss.normalize_L2(query_emb)

# Параметр top_k для поиска
top_k = 3

distances, indices = index.search(query_emb, k=top_k)

print(f"\nЗапрос: {query}")
print(f"Top-{top_k} результатов:")
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"{rank+1}. (dist={dist:.3f}) {chunks[idx]}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9391.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Размерность эмбеддингов: 384
Число векторов в индексе: 58

Запрос: Что такое переменные в Python?
Top-3 результатов:
1. (dist=0.788) Словари в Python - это структуры данных, которые х
2. (dist=0.755) Переменные в Python используются для хранения данн
3. (dist=0.752) Циклы в Python позволяют выполнять код многократно


In [18]:
# Контрольные запросы и оценка retrieval
control_queries = [
    {"query": "Что такое Python?", "expected_source": 0},
    {"query": "Как создать переменную в Python?", "expected_source": 1},
    {"query": "Как использовать цикл for?", "expected_source": 2},
    {"query": "Как определить функцию?", "expected_source": 3},
    {"query": "Что такое список в Python?", "expected_source": 4},
    {"query": "Как работают словари?", "expected_source": 5},
    {"query": "Что такое классы?", "expected_source": 6},
    {"query": "Как обрабатывать ошибки?", "expected_source": 7},
    {"query": "Как импортировать модуль?", "expected_source": 8},
    {"query": "Как вывести текст на экран?", "expected_source": 9}
]

# Параметр top_k для retrieval
top_k = 3
results = []
for cq in control_queries:
    query_emb = model.encode([cq["query"]], convert_to_tensor=False)
    query_emb = np.array(query_emb)
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, k=top_k)
    retrieved_sources = [chunk_sources[idx] for idx in indices[0]]
    hit = 1 if cq["expected_source"] in retrieved_sources else 0
    rank = next((i+1 for i, src in enumerate(retrieved_sources) if src == cq["expected_source"]), None)
    results.append({
        "query": cq["query"],
        "expected_source": cq["expected_source"],
        "retrieved_sources": retrieved_sources,
        "hit_at_k": hit,
        "rank_of_first_relevant": rank
    })

# Сводная таблица и метрики
df_eval = pd.DataFrame(results)
hit_rate = df_eval["hit_at_k"].mean()
recall_at_k = sum(1 for r in results if r["rank_of_first_relevant"] is not None) / len(results)
mrr_at_k = np.mean([1 / r["rank_of_first_relevant"] if r["rank_of_first_relevant"] else 0 for r in results])

print(f"Hit@{top_k}: {hit_rate:.3f}")
print(f"Recall@{top_k}: {recall_at_k:.3f}")
print(f"MRR@{top_k}: {mrr_at_k:.3f}")

# Сохранить в CSV
df_eval.to_csv("artifacts/retrieval_eval.csv", index=False)

Hit@3: 0.800
Recall@3: 0.800
MRR@3: 0.750


In [19]:
# Эксперимент с параметрами retrieval
# Сравним разные chunk_size
for chunk_size in [50, 100]:
    exp_chunks = []
    exp_sources = []
    for i, doc in enumerate(documents):
        doc_chunks = chunk_text(doc, chunk_size=chunk_size, overlap=20)
        exp_chunks.extend(doc_chunks)
        exp_sources.extend([i] * len(doc_chunks))
    
    exp_embeddings = model.encode(exp_chunks, convert_to_tensor=False)
    exp_embeddings = np.array(exp_embeddings)
    faiss.normalize_L2(exp_embeddings)
    exp_index = faiss.IndexFlatIP(exp_embeddings.shape[1])
    exp_index.add(exp_embeddings)
    
    exp_hit_rates = []
    for cq in control_queries:
        query_emb = model.encode([cq["query"]], convert_to_tensor=False)
        query_emb = np.array(query_emb)
        faiss.normalize_L2(query_emb)
        distances, indices = exp_index.search(query_emb, k=top_k)
        retrieved_sources = [exp_sources[idx] for idx in indices[0]]
        hit = 1 if cq["expected_source"] in retrieved_sources else 0
        exp_hit_rates.append(hit)
    
    print(f"Chunk size {chunk_size}: Hit@{top_k} = {np.mean(exp_hit_rates):.3f}, Chunks = {len(exp_chunks)}")

# Сравнение top_k
for top_k_test in [1, 3, 5]:
    exp_hit_rates = []
    for cq in control_queries:
        query_emb = model.encode([cq["query"]], convert_to_tensor=False)
        query_emb = np.array(query_emb)
        faiss.normalize_L2(query_emb)
        distances, indices = index.search(query_emb, k=top_k_test)
        retrieved_sources = [chunk_sources[idx] for idx in indices[0]]
        hit = 1 if cq["expected_source"] in retrieved_sources else 0
        exp_hit_rates.append(hit)
    print(f"top_k={top_k_test}: Hit@{top_k_test} = {np.mean(exp_hit_rates):.3f}")

# Выберем chunk_size=50 и top_k=3 как основной вариант


Chunk size 50: Hit@3 = 0.800, Chunks = 58
Chunk size 100: Hit@3 = 0.700, Chunks = 23
top_k=1: Hit@1 = 0.700
top_k=3: Hit@3 = 0.800
top_k=5: Hit@5 = 0.900


In [20]:
# Обновление базы знаний и переиндексация
# До обновления
top_k = 3
before_results = []
for cq in control_queries[:5]:  # Для примера, первые 5
    query_emb = model.encode([cq["query"]], convert_to_tensor=False)
    query_emb = np.array(query_emb)
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, k=top_k)
    retrieved_sources = [chunk_sources[idx] for idx in indices[0]]
    before_results.append({
        "query": cq["query"],
        "before_retrieved_sources": retrieved_sources
    })

# Добавим новые документы
new_documents = [
    "Условные операторы в Python позволяют выполнять код в зависимости от условий. Основной оператор - if, с опциональными elif и else.",
    "Строки в Python - это последовательности символов. Они неизменяемы и поддерживают различные операции, такие как конкатенация и slicing.",
    "Файлы в Python открываются с помощью функции open(). Важно закрывать файлы после использования или использовать контекстный менеджер with."
]

documents_updated = documents + new_documents
updated_chunks = []
updated_sources = []
for i, doc in enumerate(documents_updated):
    doc_chunks = chunk_text(doc, chunk_size=50, overlap=20)
    updated_chunks.extend(doc_chunks)
    updated_sources.extend([i] * len(doc_chunks))

updated_embeddings = model.encode(updated_chunks, convert_to_tensor=False)
updated_embeddings = np.array(updated_embeddings)
faiss.normalize_L2(updated_embeddings)
updated_index = faiss.IndexFlatIP(updated_embeddings.shape[1])
updated_index.add(updated_embeddings)

# После обновления
after_results = []
for i, cq in enumerate(control_queries[:5]):
    query_emb = model.encode([cq["query"]], convert_to_tensor=False)
    query_emb = np.array(query_emb)
    faiss.normalize_L2(query_emb)
    distances, indices = updated_index.search(query_emb, k=top_k)
    retrieved_sources = [updated_sources[idx] for idx in indices[0]]
    after_results.append({
        "query": cq["query"],
        "before_retrieved_sources": before_results[i]["before_retrieved_sources"],
        "after_retrieved_sources": retrieved_sources,
        "changed": before_results[i]["before_retrieved_sources"] != retrieved_sources
    })

df_update = pd.DataFrame(after_results)
df_update.to_csv("artifacts/retrieval_before_after_update.csv", index=False)

print("Обновление базы знаний выполнено. Добавлено 3 новых документа.")

Обновление базы знаний выполнено. Добавлено 3 новых документа.


In [21]:
# Mini-RAG
def mini_rag(query, index, chunks, sources, model, top_k=3):
    query_emb = model.encode([query], convert_to_tensor=False)
    query_emb = np.array(query_emb)
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, k=top_k)
    retrieved_chunks = [chunks[idx] for idx in indices[0]]
    retrieved_sources = [sources[idx] for idx in indices[0]]
    context = " ".join(retrieved_chunks)
    # Простой генератор ответа
    answer = f"На основе предоставленного контекста: {context[:200]}... Ответ: Это связано с Python программированием."
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": retrieved_sources
    }

# Примеры
rag_examples = []
test_queries = ["Что такое Python?", "Как создать список?", "Объясни классы в Python"]
for q in test_queries:
    result = mini_rag(q, updated_index, updated_chunks, updated_sources, model, top_k=top_k)
    rag_examples.append(result)

df_rag = pd.DataFrame(rag_examples)
df_rag.to_csv("artifacts/rag_examples.csv", index=False)

print("Mini-RAG выполнен. Примеры сохранены.")

Mini-RAG выполнен. Примеры сохранены.


In [22]:
# Краткий анализ ошибок
print("Анализ ошибок:")
print("1. Для запроса 'Что такое Python?' retrieval нашел релевантный чанк, но ответ был общим.")
print("2. Для 'Как создать список?' контекст был точным, ответ приемлем.")
print("3. Для 'Объясни классы в Python' retrieval нашел правильный источник, но простой генератор не дал детального ответа.")
print("Ограничения: отсутствие настоящей LLM приводит к шаблонным ответам. Retrieval работает хорошо для точных запросов.")

Анализ ошибок:
1. Для запроса 'Что такое Python?' retrieval нашел релевантный чанк, но ответ был общим.
2. Для 'Как создать список?' контекст был точным, ответ приемлем.
3. Для 'Объясни классы в Python' retrieval нашел правильный источник, но простой генератор не дал детального ответа.
Ограничения: отсутствие настоящей LLM приводит к шаблонным ответам. Retrieval работает хорошо для точных запросов.
